# Live lcls-ii-belt 

In [1]:
!echo $LCLS_LATTICE

/sdf/group/ad/sw/scm/repos/optics/lcls-lattice/


In [2]:
!caget ACCL:L1B:0200:AL:EACT

ACCL:L1B:0200:AL:EACT          0


In [3]:
%load_ext autoreload
%autoreload 2

from tools import isotime

from belt.tools import NpEncoder
import pandas as pd
import numpy as np

import h5py
import json
import epics

import sys
import os
import toml
from time import sleep, time
import datetime
from belt.evaluate import default_belt_merit
from belt.belt_impact import run_belt, evaluate_belt
from make_dashboard import make_dashboard

import matplotlib.pyplot as plt

import matplotlib as mpl
mpl.use('Agg')

# Nicer plotting
%config InlineBackend.figure_format = 'retina'

In [4]:
# Saving and loading
def save_pvdata(filename, pvdata, isotime):
    with h5py.File(filename, 'w') as h5:
        h5.attrs['isotime'] = np.bytes_(isotime)
        for k, v in pvdata.items():
            if isinstance(v, str):
                v =  np.bytes_(v)
            h5[k] = v 
def load_pvdata(filename):
    
    if not os.path.exists(filename):
        raise ValueError(f'H5 file does not exist: {filename} ')
    pvdata = {}
    with h5py.File(filename, 'r') as h5:
        isotime = h5.attrs['isotime']
        for k in h5:
            v = np.array(h5[k])        
            if v.dtype.char == 'S':
                v = str(v.astype(str))
            pvdata[k] = v
            
    return pvdata, isotime

## Todo 
1. read XTCAV data and generate beam
2. read Impact-T output
3. read live PV
4. generate figure from output

## Read Live PV


In [5]:
CSV = 'pv_mapping/lclsii_belt.csv'
DF = pd.read_csv(CSV)#.dropna()

PVLIST = list(DF['device_pv_name'].dropna()) 


DF

,Variable,device_pv_name,pv_unit
0,L1B_energy,ACCL:L1B:0200:AL:EACT,MV
1,L1B_amp1,ACCL:L1B:0200:AL:CO_AMPL,MV
2,L1B_amp2,ACCL:L1B:0200:AL:NEG_AMPL,MV
3,L1B_amp3,ACCL:L1B:0200:AL:POS_AMPL,MV
4,L1B_chirp,ACCL:L1B:0200:AL:CACT,MV
5,L1B_phase,ACCL:L1B:0200:AL:BETA,deg
6,HL_amplitude,ACCL:L1B:0200:AL:NOTA_AMPL,MV
7,HL_phase,ACCL:L1B:0200:AL:NOTA_PHASE,deg
8,L2B_energy,ACCL:L2B:0400:AL:EACT,MV
9,L2B_chirp,ACCL:L2B:0400:AL:CACT,MV


In [6]:
DF.loc[DF["Variable"] == "L1B_energy"] 

,Variable,device_pv_name,pv_unit
0,L1B_energy,ACCL:L1B:0200:AL:EACT,MV


In [7]:
LIVE = True
if LIVE:
    MONITOR = {pvname:epics.PV(pvname) for pvname in PVLIST}
   

In [8]:
MONITOR

{'ACCL:L1B:0200:AL:EACT': <PV 'ACCL:L1B:0200:AL:EACT', count=1, type=time_double, access=read-only>,
 'ACCL:L1B:0200:AL:CO_AMPL': <PV 'ACCL:L1B:0200:AL:CO_AMPL', count=1, type=time_double, access=read-only>,
 'ACCL:L1B:0200:AL:NEG_AMPL': <PV 'ACCL:L1B:0200:AL:NEG_AMPL', count=1, type=time_double, access=read-only>,
 'ACCL:L1B:0200:AL:POS_AMPL': <PV 'ACCL:L1B:0200:AL:POS_AMPL', count=1, type=time_double, access=read-only>,
 'ACCL:L1B:0200:AL:CACT': <PV 'ACCL:L1B:0200:AL:CACT', count=1, type=time_double, access=read-only>,
 'ACCL:L1B:0200:AL:BETA': <PV 'ACCL:L1B:0200:AL:BETA', count=1, type=time_double, access=read-only>,
 'ACCL:L1B:0200:AL:NOTA_AMPL': <PV 'ACCL:L1B:0200:AL:NOTA_AMPL', count=1, type=time_double, access=read-only>,
 'ACCL:L1B:0200:AL:NOTA_PHASE': <PV 'ACCL:L1B:0200:AL:NOTA_PHASE', count=1, type=time_double, access=read-only>,
 'ACCL:L2B:0400:AL:EACT': <PV 'ACCL:L2B:0400:AL:EACT', count=1, type=time_double, access=read-only>,
 'ACCL:L2B:0400:AL:CACT': <PV 'ACCL:L2B:0400:AL

In [9]:
def get_snapshot(snapshot_file=None):
        
    if LIVE:
        itime = isotime()
        pvdata =  {k:MONITOR[k].get() for k in MONITOR}
        
    else:
        pvdata, itime = load_pvdata(snapshot_file)
        itime = itime.decode('utf-8')
    
    #logger.info(f'Acquired settings from EPICS at: {itime}')
    
    epics_working_check = [val for val in pvdata.values() if val is None]
    
    if len(epics_working_check) == len(list(pvdata.keys())):
        raise Exception(f'EPICS returned None for all keys. Please check if you are able to connect to Accelerator')

    VCC_Key = None
    
    for k, v in pvdata.items():
        
        if v is None:
            raise ValueError(f'EPICS get for {k} returned None')
        
        if ':IMAGE:ARRAYDATA' in k.upper():
            VCC_Key = k
            found = False
            logger.info(f'Waiting for good {k}')
            counter = 0
            USE_VCC_LOCAL = True
            while not found and counter < 5:
                counter += 1
                if v is None:
                    continue
                if v.std() > 10:
                    found = True
                else:
                    v = MONITOR[k].get()
            if counter == 5:
                logger.info(f'VCC is not working. Defaulting to None.')
                USE_VCC_LOCAL = False
            elif np.ptp(v) < 128:
                v = v.astype(np.int8) # Downcast preemptively 
            pvdata[k] = v
        else:
            USE_VCC_LOCAL = False

    if not USE_VCC_LOCAL and VCC_Key in pvdata:
        del pvdata[VCC_Key]

    return pvdata, itime, USE_VCC_LOCAL

In [10]:
df = DF[DF['device_pv_name'].notna()]
assert len(df) > 0, 'Empty dataframe!'
    
pv_names = list(df['device_pv_name'])

pvdata, itime, USE_VCC_LOCAL = get_snapshot(None)
    
df['pv_value'] = [pvdata[k] for k in pv_names]

In [11]:
df

,Variable,device_pv_name,pv_unit,pv_value
0,L1B_energy,ACCL:L1B:0200:AL:EACT,MV,0.000000
1,L1B_amp1,ACCL:L1B:0200:AL:CO_AMPL,MV,102.142707
2,L1B_amp2,ACCL:L1B:0200:AL:NEG_AMPL,MV,73.402774
3,L1B_amp3,ACCL:L1B:0200:AL:POS_AMPL,MV,82.315325
4,L1B_chirp,ACCL:L1B:0200:AL:CACT,MV,0.000000
5,L1B_phase,ACCL:L1B:0200:AL:BETA,deg,-29.737417
6,HL_amplitude,ACCL:L1B:0200:AL:NOTA_AMPL,MV,52.611965
7,HL_phase,ACCL:L1B:0200:AL:NOTA_PHASE,deg,8.745729
8,L2B_energy,ACCL:L2B:0400:AL:EACT,MV,0.000000
9,L2B_chirp,ACCL:L2B:0400:AL:CACT,MV,0.000000


In [12]:
def get_settings(csv, base_settings={}, snapshot_dir=None, snapshot_file=None):
    """
    Fetches live settings for all devices in the CSV table, and translates them to simulation inputs
     
    """
    df = DF[DF['device_pv_name'].notna()]
    assert len(df) > 0, 'Empty dataframe!'
    
    pv_names = list(df['device_pv_name'])

    pvdata, itime, USE_VCC_LOCAL = get_snapshot(snapshot_file)
    
    df['pv_value'] = [pvdata[k] for k in pv_names]
    
    # Assign impact
    #df['impact_value'] = df['impact_factor']*df['pv_value'] 
    #if 'impact_offset' in df:
    #    df['impact_value'] = df['impact_value']  + df['impact_offset']

    # Collect settings
    settings = base_settings.copy()

    
    
    #HL_energy = df.loc[df["Variable"] == "HL_energy", 'pv_value' ].values[0]
    #HL_chirp = df.loc[df["Variable"] == "HL_chirp", 'pv_value' ].values[0]

    #HL_phase = df.loc[df["Variable"] == "HL_phase", 'pv_value' ].values[0]
    #HL_amplitude = np.abs(HL_energy/np.cos(HL_phase/180*np.pi)*1e6/5.5346304)

    HL_phase = df.loc[df["Variable"] == "HL_phase", 'pv_value' ].values[0] - 180
    HL_amplitude = df.loc[df["Variable"] == "HL_amplitude", 'pv_value' ].values[0]*1e6
    HL_gradient = HL_amplitude/5.5346304
    
    L1_energy = df.loc[df["Variable"] == "L1B_energy", 'pv_value' ].values[0]
    L1_chirp = df.loc[df["Variable"] == "L1B_chirp", 'pv_value' ].values[0]

    L1_phase = df.loc[df["Variable"] == "L1B_phase", 'pv_value' ].values[0]
    L1_amplitude = (df.loc[df["Variable"] == "L1B_amp1", 'pv_value' ].values[0] + 
                    df.loc[df["Variable"] == "L1B_amp2", 'pv_value' ].values[0] +
                    df.loc[df["Variable"] == "L1B_amp3", 'pv_value' ].values[0])*1e6
    L1_gradient = L1_amplitude/16.603888
                    
    #L1_amplitude = np.abs((L1_energy - HL_energy)/ np.cos(L1_phase/180*np.pi)*1e6/16.603888)
    #L1_amplitude = np.abs((L1_energy - HL_energy)*1e6/16.603888)



    L2_energy = df.loc[df["Variable"] == "L2B_energy", 'pv_value' ].values[0]
    L2_chirp = df.loc[df["Variable"] == "L2B_chirp", 'pv_value' ].values[0]

    L2_phase = df.loc[df["Variable"] == "L2B_phase", 'pv_value' ].values[0]
    L2_gradient = np.abs(L2_energy/np.cos(L2_phase/180*np.pi)*1e6/99.623328)
    
    

    L3_energy = df.loc[df["Variable"] == "L3B_energy", 'pv_value' ].values[0]
    L3_chirp = df.loc[df["Variable"] == "L3B_chirp", 'pv_value' ].values[0]

    L3_phase = df.loc[df["Variable"] == "L3B_phase", 'pv_value' ].values[0]
    L3_gradient = np.abs(L3_energy/np.cos(L3_phase/180*np.pi)*1e6/166.038878)
    #L3_amplitude = np.abs(L3_energy*1e6/166.038878)
    
    
    BC1_energy = df.loc[df["Variable"] == "BC1_energy", 'pv_value' ].values[0]/1e3
    BC1_rigidity = (df.loc[df["Variable"] == "BCX11", 'pv_value' ].values[0] + df.loc[df["Variable"] == "BCX12", 'pv_value' ].values[0] +
             df.loc[df["Variable"] == "BCX13", 'pv_value' ].values[0] + df.loc[df["Variable"] == "BCX14", 'pv_value' ].values[0])/4/10
    BC1_angle = BC1_rigidity/(3.3356*BC1_energy)

    BC2_energy = df.loc[df["Variable"] == "BC2_energy", 'pv_value' ].values[0]/1e3
    BC2_rigidity = (df.loc[df["Variable"] == "BCX21", 'pv_value' ].values[0] + df.loc[df["Variable"] == "BCX22", 'pv_value' ].values[0] +
             df.loc[df["Variable"] == "BCX23", 'pv_value' ].values[0] + df.loc[df["Variable"] == "BCX24", 'pv_value' ].values[0])/4/10
    BC2_angle = BC2_rigidity/(3.3356*BC2_energy)


    BC1_energy_increment =BC1_energy*1e9 - (90e6 + L1_amplitude*np.cos(L1_phase/180*np.pi) +  HL_amplitude*np.cos(HL_phase/180*np.pi))
    #BC1_energy_increment =BC1_energy*1e9 - (90e6 + L1_energy*1e6 )
    BC2_energy_increment = BC2_energy*1e9 - (BC1_energy*1e9 + L2_energy*1e6)

    settings["BC1:angle"] = BC1_angle
    settings["BC2:angle"] = BC2_angle
    settings["L1:gradient"] = L1_gradient
    settings["L1:phase_deg"] = L1_phase 
    settings["L2:gradient"] = L2_gradient
    settings["L2:phase_deg"] = L2_phase
    settings["L3:gradient"] = L3_gradient
    settings["L3:phase_deg"] = L3_phase 
    settings["HL:gradient"] = HL_gradient
    settings["HL:phase_deg"] = HL_phase 
    settings["EBC1:energy_increment"] = BC1_energy_increment
    settings["EBC2:energy_increment"] = BC2_energy_increment
    
    #if DEBUG:
    #    settings['total_charge'] = 0
    #else:
    #    settings['total_charge'] = 1 # Will be updated with particles

    # VCC image
    #if USE_VCC_LOCAL:
    #    logger.info('Getting VCC Live Distgen')
    #    dfile, img, cutimg = get_live_distgen_xy_dist(filename=DISTGEN_LASER_FILE, vcc_device=VCC_DEVICE, pvdata=pvdata)  
    #    settings['distgen:xy_dist:file'] = dfile
    #elif USE_SAVED_VCC:
    #    settings['distgen:xy_dist:file'] = SAVED_VCC
    #    img, cutimg = None, None
    #else:
    #    img, cutimg = None, None
        #settings['distgen:r_dist:max_r:value'] = 0.35 # TEMP     
        
    if snapshot_dir and not snapshot_file:
        filename = os.path.abspath(os.path.join(snapshot_dir, f'{MODEL}-snapshot-{itime}.h5'))
    #    total_charge_pC = settings['distgen:total_charge:value']
    #    if total_charge_pC < MIN_CHARGE_pC:
    #        logger.info(f'total charge is too low: {total_charge_pC:.2f} pC, not saving snapshot')         
    #    else:
        save_pvdata(filename, pvdata, itime)
    #        logger.info(f'EPICS shapshot written: {filename}')
        
        
    return settings, df, itime

In [13]:
# Patch this into the function below for the dashboard creation
def my_merit(belt_object, itime):
    # Collect standard output statistics
    merit0 = default_belt_merit(belt_object)
    
    PLOT_OUTPUT_DIR_DATED = convertToDatedFormat(PLOT_OUTPUT_DIR)
    #Overriding at runtime to save in dated folders
    DASHBOARD_KWARGS["outpath"] = PLOT_OUTPUT_DIR_DATED
    
    # Make the dashboard from the evaluated object
    plot_file = make_dashboard(belt_object, itime=itime, **DASHBOARD_KWARGS)
    #print('Dashboard written:', plot_file)
    #logger.info(f'Dashboard written: {plot_file}')
    
    # Make all readable
    os.chmod(plot_file, 0o644)
    
    # Assign extra info
    merit0['plot_file'] = plot_file    
    merit0['isotime'] = itime
    
    # Clear any buffers
    plt.close('all')

    return merit0

In [14]:
def convertToDatedFormat(destionation_folder):
    curr_date = datetime.date.today()
    year,month,day = curr_date.strftime('%Y'),curr_date.strftime('%m'),curr_date.strftime('%d')
    destionation_folder_dated = destionation_folder + "/" + year + "/" + month + "/" + day

    if not os.path.exists(destionation_folder_dated):
        os.makedirs(destionation_folder_dated)
    
    return destionation_folder_dated

In [15]:
dat = {}

MODEL = 'LCLSII'
HOST = 's3df'
ARCHIVE_DIR = './archive'

SNAPSHOT_DIR = './snapshot'
SUMMARY_OUTPUT_DIR = './summary'
PLOT_OUTPUT_DIR = './plot'
SETTINGS0 = {"Impact_particles": "/sdf/data/ad/ard/u/jytang/lume-belt-live/impact_particles/final_particles.h5", 
             "num_doublings": 7}
CONFIG0 = {"input": "example1/belt.in", "workdir": os.environ.get("SCRATCH")}
PREFIX = f'lume-belt-live-demo-{HOST}-{MODEL}'


DASHBOARD_KWARGS = {'outpath':PLOT_OUTPUT_DIR,            
                    'name' : PREFIX
                   }    

SNAPSHOT = None
SNAPSHOT_DIR_DATED = convertToDatedFormat(SNAPSHOT_DIR)
ARCHIVE_DIR_DATED = convertToDatedFormat(ARCHIVE_DIR)
SUMMARY_OUTPUT_DIR_DATED = convertToDatedFormat(SUMMARY_OUTPUT_DIR)
settings, df, itime = get_settings(CSV,
                                                           SETTINGS0,
                                                           snapshot_dir=SNAPSHOT_DIR_DATED,
                                                          snapshot_file=SNAPSHOT)       

In [16]:
9171296.148254855/1e6*5.5346304*np.cos(-171.46964410206522/180*np.pi)

np.float64(-50.19820052129489)

In [17]:
settings

{'Impact_particles': '/sdf/data/ad/ard/u/jytang/lume-belt-live/impact_particles/final_particles.h5',
 'num_doublings': 7,
 'BC2:angle': np.float64(0.04682866448132572),
 'L1:gradient': np.float64(15530146.040099064),
 'L1:phase_deg': np.float64(-29.737417118442867),
 'L2:gradient': np.float64(0.0),
 'L2:phase_deg': np.float64(-19.103097013379756),
 'L3:gradient': np.float64(0.0),
 'L3:phase_deg': np.float64(0.0),
 'HL:gradient': np.float64(9505958.183879474),
 'HL:phase_deg': np.float64(-171.25427098941302),
 'EBC1:energy_increment': np.float64(-11893516.01718384),
 'EBC2:energy_increment': np.float64(1250066207.5952399)}

In [18]:
def run1():
    dat = {}

    SNAPSHOT_DIR_DATED = convertToDatedFormat(SNAPSHOT_DIR)
    ARCHIVE_DIR_DATED = convertToDatedFormat(ARCHIVE_DIR)
    SUMMARY_OUTPUT_DIR_DATED = convertToDatedFormat(SUMMARY_OUTPUT_DIR)
        
    # Acquire settings
    mysettings, df,  itime = get_settings(CSV,
                                                           SETTINGS0,
                                                           snapshot_dir=SNAPSHOT_DIR_DATED,
                                                          snapshot_file=SNAPSHOT)        
    print(mysettings)
    dat['isotime'] = itime
    
    # Record inputs
    dat['inputs'] = mysettings
    dat['config'] = CONFIG0
    dat['pv_mapping_dataframe'] = df.to_dict()
    
    #logger.info(f'Running evaluate_impact_with_distgen...')

    t0 = time()
    
    #total_charge_pC = mysettings['distgen:total_charge:value']
    #if total_charge_pC < MIN_CHARGE_pC:
    #    logger.info(f'total charge is too low: {total_charge_pC:.2f} pC, skipping')
    #    return dat
    
    outputs = evaluate_belt(CONFIG0, mysettings,
                                       merit_f=lambda x: my_merit(x, itime),
                                       archive_path=ARCHIVE_DIR_DATED,
                                        verbose=True )
    
    dat['outputs'] =  outputs   
    #logger.info(f'...finished in {(time()-t0):.1f} s')
    fname = fname=f'{SUMMARY_OUTPUT_DIR_DATED}/{PREFIX}-{itime}.json'

    json.dump(dat, open(fname, 'w'), cls=NpEncoder)
    #logger.info(f'Summary output written: {fname}')
    return dat
    

In [19]:
result = run1()

{'Impact_particles': '/sdf/data/ad/ard/u/jytang/lume-belt-live/impact_particles/final_particles.h5', 'num_doublings': 7, 'BC2:angle': np.float64(0.04682872940763924), 'L1:gradient': np.float64(15530146.040099064), 'L1:phase_deg': np.float64(-29.737417118442867), 'L2:gradient': np.float64(0.0), 'L2:phase_deg': np.float64(-19.103097013379756), 'L3:gradient': np.float64(0.0), 'L3:phase_deg': np.float64(0.0), 'HL:gradient': np.float64(9505958.183879474), 'HL:phase_deg': np.float64(-171.25427098941302), 'EBC1:energy_increment': np.float64(-11893516.01718384), 'EBC2:energy_increment': np.float64(1250066207.5952399)}
Reading Impact_particles = /sdf/data/ad/ard/u/jytang/lume-belt-live/impact_particles/final_particles.h5
Upsampling the particle number to  1279746
Setting BELT BC2:angle = 0.04682872940763924
Setting BELT L1:gradient = 15530146.040099064
Setting BELT L1:phase_deg = -29.737417118442867
Setting BELT L2:gradient = 0.0
Setting BELT L2:phase_deg = -19.103097013379756
Setting BELT L3:g

<!-- lume-genesis detected Jupyter and will use HTML for rendering. -->

In [19]:
result['outputs']['archive']

'/sdf/data/ad/ard/u/jytang/lume-belt-live/archive/2024/12/06/fa3f1f7cf76048c50bcd9fefbe2ecb0a.h5'

In [55]:
#Sanity Checks for OS Environments
if 'LCLS_LATTICE' in os.environ:
    print('LCLS Lattice is set to - ', os.environ['LCLS_LATTICE'])
else:
    print('LCLS_LATTICE Location is missing')
    exit(1)
if 'LUME_OUTPUT_FOLDERS' in os.environ:
    print('LUME_OUTPUT_FOLDERS is set to - ', os.environ['LUME_OUTPUT_FOLDERS'])
else:
    print('LUME_OUTPUT_FOLDERS Location is missing')
    exit(1)
if 'SCRATCH' in os.environ:
    print('SCRATCH is set to - ', os.environ['SCRATCH'])
else:
    print('SCRATCH Location is missing')
    exit(1)

def replaceEnvironmentFiles(file_location):
    if 'LUME_OUTPUT_FOLDERS' in file_location:
        return file_location.replace('$LUME_OUTPUT_FOLDERS', os.environ['LUME_OUTPUT_FOLDERS'])
    if 'LCLS_LATTICE' in file_location:
        return file_location.replace('$LCLS_LATTICE', os.environ['LCLS_LATTICE'])
    if 'SCRATCH' in file_location:
        return file_location.replace('$SCRATCH', os.environ['SCRATCH'])
    return file_location

LCLS Lattice is set to -  /sdf/group/ad/sw/scm/repos/optics/lcls-lattice/
LUME_OUTPUT_FOLDERS is set to -  /sdf/data/ad/ard/u/jytang/lume-impact-live/
SCRATCH is set to -  /sdf/scratch/users/j/jytang


# Top level config

In [56]:
import argparse

parser = argparse.ArgumentParser()
parser.add_argument("-d", "--debug", help = "Debug Mode", default = False)
parser.add_argument("-v", "--use_vcc", help = "Use VCC - True When VCC is Active", default = True)
parser.add_argument("-l", "--live", help = "Live Mode -  True When BEAM is Active", default = True)
parser.add_argument("-m", "--model", help = "Mention the Injector Model", default = "sc_inj")
parser.add_argument("-t", "--host", help = "Mention the host", default = "singularity")
parser.add_argument("-p", "--num_procs", help = "Mention the Num Procs", default = 64)

_StoreAction(option_strings=['-p', '--num_procs'], dest='num_procs', nargs=None, const=None, default=64, type=None, choices=None, required=False, help='Mention the Num Procs', metavar=None)

In [57]:
def convertStringToBoolean(argument):
    if argument == 'True' or argument == 'true' or argument == True:
        return True
    else:
        return False

In [58]:
#%tb
#args = vars(parser.parse_args())

#DEBUG = convertStringToBoolean(args['debug'])
#USE_VCC = convertStringToBoolean(args['use_vcc'])
#LIVE = convertStringToBoolean(args['live'])
#MODEL = args['model']
#HOST = args['host']
#NUM_PROCS_ARGS = int(args['num_procs'])

DEBUG = False
USE_VCC = True
USE_SAVED_VCC = False
SAVED_VCC = "/sdf/group/ad/beamphysics/jytang/lume-impact-live-demo/configs/vcc_image/laser_06032024.txt"
LIVE = True
MODEL = "sc_inj"
HOST = 's3df'
NUM_PROCS_ARGS = 64

SNAPSHOT = 'examples/sc_inj-snapshot-2022-11-12T12:38:08-08:00.h5'
MIN_CHARGE_pC = 10
config = toml.load(f"configs/{HOST}_{MODEL}.toml")
PREFIX = f'lume-impact-live-demo-{HOST}-{MODEL}'

In [59]:
def convertToDatedFormat(destionation_folder):
    curr_date = datetime.date.today()
    year,month,day = curr_date.strftime('%Y'),curr_date.strftime('%m'),curr_date.strftime('%d')
    destionation_folder_dated = destionation_folder + "/" + year + "/" + month + "/" + day

    if not os.path.exists(destionation_folder_dated):
        os.makedirs(destionation_folder_dated)
    
    return destionation_folder_dated

## Logging

In [60]:
import logging
from logging.handlers import RotatingFileHandler

# Gets or creates a logger
logger = logging.getLogger(PREFIX)  

# set log level
logger.setLevel(logging.INFO)

LOG_OUTPUT_DIR = config.get("log_output_dir")
LOG_OUTPUT_DIR = replaceEnvironmentFiles(LOG_OUTPUT_DIR)
# define file handler and set formatter
file_handler = RotatingFileHandler(f'{LOG_OUTPUT_DIR}/{PREFIX}.log', mode='a', encoding=None, maxBytes=50*1024*1024, 
                                 backupCount=2, delay=0)
formatter    = logging.Formatter(fmt="%(asctime)s :  %(name)s : %(message)s ", datefmt="%Y-%m-%dT%H:%M:%S%z")

# Add print to stdout
logger.addHandler(logging.StreamHandler(sys.stdout))

file_handler.setFormatter(formatter)

# add file handler to logger
logger.addHandler(file_handler)

In [61]:
#Arguments -

logger.info('Start of Script Marker - Script Running with Arguments - ')
logger.info(f'Debug - {DEBUG}')
logger.info(f'USE_VCC - {USE_VCC}')
logger.info(f'LIVE - {LIVE}')
logger.info(f'MODEL - {MODEL}')
logger.info(f'HOST - {HOST}')
logger.info(f'NUM_PROCS_ARGS - {NUM_PROCS_ARGS}')
logger.info(f'Config TOML Loaded - {config}')

Start of Script Marker - Script Running with Arguments - 
Start of Script Marker - Script Running with Arguments - 
Start of Script Marker - Script Running with Arguments - 
Debug - False
Debug - False
Debug - False
USE_VCC - True
USE_VCC - True
USE_VCC - True
LIVE - True
LIVE - True
LIVE - True
MODEL - sc_inj
MODEL - sc_inj
MODEL - sc_inj
HOST - s3df
HOST - s3df
HOST - s3df
NUM_PROCS_ARGS - 64
NUM_PROCS_ARGS - 64
NUM_PROCS_ARGS - 64
Config TOML Loaded - {'host': 'sdf', 'config_file': '/$LCLS_LATTICE/impact/models/sc_inj/v1/ImpactT.yaml', 'distgen_input_file': '/$LCLS_LATTICE/distgen/models/sc_inj/vcc_image/distgen.yaml', 'workdir': '$SCRATCH', 'summary_output_dir': '/$LUME_OUTPUT_FOLDERS/summary', 'plot_output_dir': '/$LUME_OUTPUT_FOLDERS/plot', 'log_output_dir': '/$LUME_OUTPUT_FOLDERS/log', 'archive_dir': '/$LUME_OUTPUT_FOLDERS/archive', 'snapshot_dir': '/$LUME_OUTPUT_FOLDERS/snapshot', 'scan_output_dir': '/$LUME_OUTPUT_FOLDERS/scan', 'distgen_laser_file': '/sdf/group/ad/beamphysics/

## Utils

In [33]:
# Saving and loading
def save_pvdata(filename, pvdata, isotime):
    with h5py.File(filename, 'w') as h5:
        h5.attrs['isotime'] = np.bytes_(isotime)
        for k, v in pvdata.items():
            if isinstance(v, str):
                v =  np.bytes_(v)
            h5[k] = v 
def load_pvdata(filename):
    
    if not os.path.exists(filename):
        raise ValueError(f'H5 file does not exist: {filename} ')
    pvdata = {}
    with h5py.File(filename, 'r') as h5:
        isotime = h5.attrs['isotime']
        for k in h5:
            v = np.array(h5[k])        
            if v.dtype.char == 'S':
                v = str(v.astype(str))
            pvdata[k] = v
            
    return pvdata, isotime

# Configuration

Set up basic input sources and output path, loaded from toml environment file.

See README for required toml definition.

In [34]:
HOST = config.get('host') # mcc-simul or 'sdf'
if not HOST:
    raise ValueError("host not defined in toml.")
    
def get_path(key):
    val = config.get(key)
    if not val:
        raise ValueError(f"{key} not defined in toml.")
    val=os.path.expandvars(val)
    if not os.path.exists(val):
        raise ValueError(f"{val} does not exist")
    return os.path.abspath(val)


# Output dirs

SUMMARY_OUTPUT_DIR = replaceEnvironmentFiles(get_path('summary_output_dir'))
ARCHIVE_DIR = replaceEnvironmentFiles(get_path('archive_dir'))
SNAPSHOT_DIR = replaceEnvironmentFiles(get_path('snapshot_dir'))

# Dummy file for distgen
DISTGEN_LASER_FILE = config.get('distgen_laser_file')
if not DISTGEN_LASER_FILE:
    raise ValueError("distgen_laser_file not defined in toml.")

# Number of processors
NUM_PROCS = config.get('num_procs')
if not NUM_PROCS:
    raise ValueError("num_procs not defined in toml.")
else:
    NUM_PROCS = int(NUM_PROCS)

if NUM_PROCS_ARGS != NUM_PROCS:
    NUM_PROCS = NUM_PROCS_ARGS

# if using sdf:
if HOST == 'sdf':    
    #check that environment variables are configured for execution
    IMPACT_COMMAND = config.get("impact_command")
    if not IMPACT_COMMAND:
       raise ValueError("impact_command not defined in toml.")


    IMPACT_COMMAND_MPI = config.get("impact_command_mpi")
    if not IMPACT_COMMAND_MPI:
       raise ValueError("impact_command_mpi not defined in toml.")



In [35]:
CONFIG0 = {}

# Base settings
SETTINGS0 = {
 'distgen:n_particle': 10_000,   
 'timeout': 10000,
 'header:Nx': 32,
 'header:Ny': 32,
 'header:Nz': 32,
 'numprocs': NUM_PROCS,
   }

SETTINGS0['numprocs'] = NUM_PROCS
CONFIG0["workdir"] = replaceEnvironmentFiles(get_path('workdir'))

if DEBUG:
    logger.info('DEBUG MODE: Running without space charge for speed. ')
    SETTINGS0['distgen:n_particle'] = 1000
    SETTINGS0['total_charge'] = 0
    
# Host config    
if HOST in ('sdf'):
    
    #SDF setup 
    SETTINGS0['command'] =  IMPACT_COMMAND
    SETTINGS0['command_mpi'] =  IMPACT_COMMAND_MPI
    SETTINGS0['mpi_run'] = config.get("mpi_run_cmd")
    
elif HOST == 'local':
    logger.info('Running locally')
    
else:
    raise ValueError(f'Unknown host: {HOST}')
    

# Select: LCLS or FACET

In [36]:
# PV -> Sim conversion table
CSV =  f'pv_mapping/{MODEL}_impact.csv'  

CONFIG0['impact_config']      =  replaceEnvironmentFiles(get_path('config_file'))
CONFIG0['distgen_input_file'] =  replaceEnvironmentFiles(get_path('distgen_input_file'))

print('Impact Config Loaded - ', CONFIG0['impact_config'] )
print('Distgen Input File Loaded - ', CONFIG0['distgen_input_file'] )

PLOT_OUTPUT_DIR = replaceEnvironmentFiles(get_path('plot_output_dir'))

if MODEL == 'cu_inj':
    VCC_DEVICE = 'CAMR:IN20:186' # LCLS   
    
    DASHBOARD_KWARGS = {'outpath':PLOT_OUTPUT_DIR,
                    'screen1': 'YAG02',
                    'screen2': 'YAG03',
                    'screen3': 'OTR2',
                    'ylim' : (0, None), # Emittance scale   
                    'ylim2': (0, None), # sigma_x scale
                    'name' : PREFIX
                   }    
    
    SETTINGS0['stop'] = 16.5
    SETTINGS0['distgen:t_dist:length:value'] =  4 * 1.65   #  Inferred pulse stacker FWHM: 4 ps, converted to tukey length
    
if MODEL == 'sc_inj':
    VCC_DEVICE = 'CAMR:LGUN:950' # LCLS-II 
    
    DASHBOARD_KWARGS = {'outpath':PLOT_OUTPUT_DIR,
                    'screen1': 'YAG01B',
                  #  'screen2': 'BEAM0',
                  #  'screen3': 'OTR0H04',
                    'screen2': 'CM01BEG',
                    'screen3': 'BEAM0',
                    'ylim' : (0, 3e-6), # Emittance scale   
                    'ylim2': (0, None), # sigma_x scale                    
                    'name' : PREFIX
                   }    
    
    SETTINGS0['stop'] = 14 # 28
    SETTINGS0['distgen:t_dist:sigma_t:value'] =  16 / 2.355   # ps, equivalent to 16ps FWHM from Feng
    
elif MODEL == 'f2e_inj':
    VCC_DEVICE = 'CAMR:LT10:900' # FACET-II
    
    DASHBOARD_KWARGS = {'outpath':PLOT_OUTPUT_DIR,
                    'screen1': 'PR10241',
                    'screen2': 'PR10465',
                    'screen3': 'PR10571',
                    'ylim' : (0, 20e-6), # Emittance scale
                    'name' : PREFIX
                   }        
    
    SETTINGS0['distgen:t_dist:length:value'] =  3.65 * 1.65   #  Measured FWHM: 3.65 ps, converted to tukey length
     
else:
    raise

Impact Config Loaded -  //sdf/group/ad/sw/scm/repos/optics/lcls-lattice/impact/models/sc_inj/v1/ImpactT.yaml
Distgen Input File Loaded -  //sdf/group/ad/sw/scm/repos/optics/lcls-lattice/distgen/models/sc_inj/vcc_image/distgen.yaml


In [37]:
CONFIG0, SETTINGS0
logger.info(f'FINAL SETTINGS - {SETTINGS0}')

FINAL SETTINGS - {'distgen:n_particle': 10000, 'timeout': 10000, 'header:Nx': 32, 'header:Ny': 32, 'header:Nz': 32, 'numprocs': 64, 'command': '~/miniconda3/envs/lume-live-dev/bin/ImpactTexe', 'command_mpi': '~/miniconda3/envs/lume-live-dev/bin/ImpactTexe-mpi', 'mpi_run': 'salloc --partition milano --account ad:beamphysics -N 1 -n {nproc} mpirun -n {nproc} {command_mpi}', 'stop': 14, 'distgen:t_dist:sigma_t:value': 6.794055201698514}
FINAL SETTINGS - {'distgen:n_particle': 10000, 'timeout': 10000, 'header:Nx': 32, 'header:Ny': 32, 'header:Nz': 32, 'numprocs': 64, 'command': '~/miniconda3/envs/lume-live-dev/bin/ImpactTexe', 'command_mpi': '~/miniconda3/envs/lume-live-dev/bin/ImpactTexe-mpi', 'mpi_run': 'salloc --partition milano --account ad:beamphysics -N 1 -n {nproc} mpirun -n {nproc} {command_mpi}', 'stop': 14, 'distgen:t_dist:sigma_t:value': 6.794055201698514}


# Set up monitors

In [38]:
# Gun: 700 kV
# Buncher: 200 keV energy gain
# Buncher: +60 deg relative to on-crest

In [39]:
DF = pd.read_csv(CSV)#.dropna()

PVLIST = list(DF['device_pv_name'].dropna()) 

if USE_VCC:
    PVLIST = PVLIST + list(VCC_DEVICE_PV[VCC_DEVICE].values())
else:
    logger.info('USE VCC set to False. VCC is not working right now.')
#DF.set_index('device_pv_name', inplace=True)
DF

,Variable,bmad_name,device_pv_name,pv_unit,impact_name,impact_factor,impact_offset,impact_description,impact_unit
0,Gun Voltage,RFGUNB,SOLN:GUNB:212:BACT,xxx,RFGUNB:rf_field_scale,0.000000e+00,1.764633e+07,Value for 670 kV,V/m
1,Gun phase,RFGUNB,SOLN:GUNB:212:BACT,xxx,RFGUNB:autophase_deg,0.000000e+00,0.000000e+00,NaN,deg
2,Solenoid 1,SOL1B,SOLN:GUNB:212:BACT,kG*m,SOL1B:solenoid_field_scale,1.159555e+00,0.000000e+00,peak field,T
3,Solenoid 2,SOL2B,SOLN:GUNB:823:BACT,kG*m,SOL2B:solenoid_field_scale,1.159555e+00,0.000000e+00,peak field,T
4,charge,CATHODE,BPMS:GUNB:314:TMIT,n_electrons,distgen:total_charge:value,1.600000e-07,0.000000e+00,total charge on cathode,pC
5,Buncher Voltage,BUN1B,ACCL:GUNB:455:AACT_AVG,MV,BUN1B:rf_field_scale,0.000000e+00,1.786301e+06,values for 200 kV,V/m
6,Buncher Phase,BUN1B,ACCL:GUNB:455:PACT_AVG,deg,BUN1B:autophase_deg,0.000000e+00,-6.000000e+01,phase relative to on-crest,deg
7,Cavity 1 voltage,CAVL011,ACCL:L0B:0110:AACTMEAN,MV,CAVL011:rf_field_scale,1.861947e+06,0.000000e+00,peak on-axis electric field,V/m
8,Cavity 2 voltage,CAVL012,ACCL:L0B:0120:AACTMEAN,MV,CAVL012:rf_field_scale,1.861947e+06,0.000000e+00,peak on-axis electric field,V/m
9,Cavity 3 voltage,CAVL013,ACCL:L0B:0130:AACTMEAN,MV,CAVL013:rf_field_scale,1.861947e+06,0.000000e+00,peak on-axis electric field,V/m


In [40]:
if LIVE:
    MONITOR = {pvname:epics.PV(pvname) for pvname in PVLIST}
    MONITOR.pop('ACCL:GUNB:455:AACT_AVG', None)
    MONITOR.pop('ACCL:GUNB:455:PACT_AVG', None)
    SNAPSHOT = None
    sleep(5)

In [41]:
buncher_voltage = 190
variablex = "Buncher Voltage"
DF.loc[DF["Variable"] == variablex, 'impact_offset' ] = 1786301.125*buncher_voltage/200
DF.loc[DF["Variable"] == variablex, 'impact_description' ] = f'Value for {buncher_voltage} kV'

variablex = "Gun Voltage"
gun_energy = 650
DF.loc[DF["Variable"] == variablex, 'impact_offset' ] = 16994214.42418604*gun_energy/670
DF.loc[DF["Variable"] == variablex, 'impact_description' ] = f'Value for {gun_energy} kV'

variablex = "Buncher Phase"
DF.loc[DF["Variable"] == variablex, 'impact_offset' ] = -57


In [42]:
def get_snapshot(snapshot_file=None):
        
    if LIVE:
        itime = isotime()
        pvdata =  {k:MONITOR[k].get() for k in MONITOR}
        pvdata['ACCL:GUNB:455:AACT_AVG'] = 0.0
        pvdata['ACCL:GUNB:455:PACT_AVG'] = 0.0
    else:
        pvdata, itime = load_pvdata(snapshot_file)
        itime = itime.decode('utf-8')
    
    logger.info(f'Acquired settings from EPICS at: {itime}')
    
    epics_working_check = [val for val in pvdata.values() if val is None]
    
    if len(epics_working_check) == len(list(pvdata.keys())):
        raise Exception(f'EPICS returned None for all keys. Please check if you are able to connect to Accelerator')

    VCC_Key = None
    
    for k, v in pvdata.items():
        
        if v is None:
            raise ValueError(f'EPICS get for {k} returned None')
        
        if ':IMAGE:ARRAYDATA' in k.upper():
            VCC_Key = k
            found = False
            logger.info(f'Waiting for good {k}')
            counter = 0
            USE_VCC_LOCAL = True
            while not found and counter < 5:
                counter += 1
                if v is None:
                    continue
                if v.std() > 10:
                    found = True
                else:
                    v = MONITOR[k].get()
            if counter == 5:
                logger.info(f'VCC is not working. Defaulting to None.')
                USE_VCC_LOCAL = False
            elif np.ptp(v) < 128:
                v = v.astype(np.int8) # Downcast preemptively 
            pvdata[k] = v
        else:
            USE_VCC_LOCAL = False

    if not USE_VCC_LOCAL and VCC_Key in pvdata:
        del pvdata[VCC_Key]

    return pvdata, itime, USE_VCC_LOCAL

# EPICS -> Simulation settings

In [43]:
VCC_DEVICE

'CAMR:LGUN:950'

In [44]:
def get_settings(csv, base_settings={}, snapshot_dir=None, snapshot_file=None):
    """
    Fetches live settings for all devices in the CSV table, and translates them to simulation inputs
     
    """
    df = DF[DF['device_pv_name'].notna()]
    assert len(df) > 0, 'Empty dataframe!'
    
    pv_names = list(df['device_pv_name'])

    pvdata, itime, USE_VCC_LOCAL = get_snapshot(snapshot_file)
    
    df['pv_value'] = [pvdata[k] for k in pv_names]
    
    # Assign impact
    df['impact_value'] = df['impact_factor']*df['pv_value'] 
    if 'impact_offset' in df:
        df['impact_value'] = df['impact_value']  + df['impact_offset']

    # Collect settings
    settings = base_settings.copy()
    settings.update(dict(zip(df['impact_name'], df['impact_value'])))
    
    if DEBUG:
        settings['total_charge'] = 0
    else:
        settings['total_charge'] = 1 # Will be updated with particles

    # VCC image
    if USE_VCC_LOCAL:
        logger.info('Getting VCC Live Distgen')
        dfile, img, cutimg = get_live_distgen_xy_dist(filename=DISTGEN_LASER_FILE, vcc_device=VCC_DEVICE, pvdata=pvdata)  
        settings['distgen:xy_dist:file'] = dfile
    elif USE_SAVED_VCC:
        settings['distgen:xy_dist:file'] = SAVED_VCC
        img, cutimg = None, None
    else:
        img, cutimg = None, None
        #settings['distgen:r_dist:max_r:value'] = 0.35 # TEMP     
        
    if snapshot_dir and not snapshot_file:
        filename = os.path.abspath(os.path.join(snapshot_dir, f'{MODEL}-snapshot-{itime}.h5'))
        total_charge_pC = settings['distgen:total_charge:value']
        if total_charge_pC < MIN_CHARGE_pC:
            logger.info(f'total charge is too low: {total_charge_pC:.2f} pC, not saving snapshot')         
        else:
            save_pvdata(filename, pvdata, itime)
            logger.info(f'EPICS shapshot written: {filename}')
        
        
    return settings, df, img, cutimg, itime

In [45]:
DO_TIMING = False

if DO_TIMING:
    import numpy as np
    import time
    results = []
    tlist = []
    nlist = 2**np.arange(1,8, 1)[::-1]
    for n in nlist:
        t1 = time.time()
        LIVE_SETTINGS['numprocs'] = n
        print(f'running wit {n}')
        result = run_impact_with_distgen(LIVE_SETTINGS, **CONFIG0, verbose=False )
        results.append(result)
        dt = time.time() - t1
        tlist.append(dt)
        print(n, dt)     
        
    tlist, nlist        

# Get live values, run Impact-T, make dashboard

In [46]:
# Patch this into the function below for the dashboard creation
def my_merit(impact_object, itime):
    # Collect standard output statistics
    merit0 = default_impact_merit(impact_object)
    
    PLOT_OUTPUT_DIR_DATED = convertToDatedFormat(PLOT_OUTPUT_DIR)
    #Overriding at runtime to save in dated folders
    DASHBOARD_KWARGS["outpath"] = PLOT_OUTPUT_DIR_DATED
    
    # Make the dashboard from the evaluated object
    plot_file = make_dashboard(impact_object, itime=itime, **DASHBOARD_KWARGS)
    #print('Dashboard written:', plot_file)
    logger.info(f'Dashboard written: {plot_file}')
    
    # Make all readable
    os.chmod(plot_file, 0o644)
    
    # Assign extra info
    merit0['plot_file'] = plot_file    
    merit0['isotime'] = itime
    
    # Clear any buffers
    plt.close('all')

    return merit0

In [47]:
dat = {}

SNAPSHOT_DIR_DATED = convertToDatedFormat(SNAPSHOT_DIR)
ARCHIVE_DIR_DATED = convertToDatedFormat(ARCHIVE_DIR)
SUMMARY_OUTPUT_DIR_DATED = convertToDatedFormat(SUMMARY_OUTPUT_DIR)
mysettings, df, img, cutimg, itime = get_settings(CSV,
                                                           SETTINGS0,
                                                           snapshot_dir=SNAPSHOT_DIR_DATED,
                                                          snapshot_file=SNAPSHOT)        

Acquired settings from EPICS at: 2022-11-12T12:38:08-08:00
Acquired settings from EPICS at: 2022-11-12T12:38:08-08:00


/lscratch/jytang/tmp/ipykernel_812433/3072082356.py:17: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments.
  v = np.array(h5[k])


In [48]:
mysettings

{'distgen:n_particle': 10000,
 'timeout': 10000,
 'header:Nx': 32,
 'header:Ny': 32,
 'header:Nz': 32,
 'numprocs': 64,
 'command': '~/miniconda3/envs/lume-live-dev/bin/ImpactTexe',
 'command_mpi': '~/miniconda3/envs/lume-live-dev/bin/ImpactTexe-mpi',
 'mpi_run': 'salloc --partition milano --account ad:beamphysics -N 1 -n {nproc} mpirun -n {nproc} {command_mpi}',
 'stop': 14,
 'distgen:t_dist:sigma_t:value': 6.794055201698514,
 'RFGUNB:rf_field_scale': np.float64(16486924.441374516),
 'RFGUNB:autophase_deg': np.float64(0.0),
 'SOL1B:solenoid_field_scale': np.float64(0.05449666295586827),
 'SOL2B:solenoid_field_scale': np.float64(0.028757730622052658),
 'distgen:total_charge:value': np.float64(50.07263744),
 'BUN1B:rf_field_scale': np.float64(1696986.06875),
 'BUN1B:autophase_deg': np.float64(-57.0),
 'CAVL011:rf_field_scale': np.float64(13983964.516374521),
 'CAVL012:rf_field_scale': np.float64(3360.773170767213),
 'CAVL013:rf_field_scale': np.float64(13051841.410405656),
 'CAVL014:rf_

In [49]:
def run1():
    dat = {}

    SNAPSHOT_DIR_DATED = convertToDatedFormat(SNAPSHOT_DIR)
    ARCHIVE_DIR_DATED = convertToDatedFormat(ARCHIVE_DIR)
    SUMMARY_OUTPUT_DIR_DATED = convertToDatedFormat(SUMMARY_OUTPUT_DIR)
        
    # Acquire settings
    mysettings, df, img, cutimg, itime = get_settings(CSV,
                                                           SETTINGS0,
                                                           snapshot_dir=SNAPSHOT_DIR_DATED,
                                                          snapshot_file=SNAPSHOT)        
    dat['isotime'] = itime
    
    # Record inputs
    dat['inputs'] = mysettings
    dat['config'] = CONFIG0
    dat['pv_mapping_dataframe'] = df.to_dict()
    
    logger.info(f'Running evaluate_impact_with_distgen...')

    t0 = time()
    
    total_charge_pC = mysettings['distgen:total_charge:value']
    if total_charge_pC < MIN_CHARGE_pC:
        logger.info(f'total charge is too low: {total_charge_pC:.2f} pC, skipping')
        return dat
    
    outputs = evaluate_impact_with_distgen(mysettings,
                                       merit_f=lambda x: my_merit(x, itime),
                                       archive_path=ARCHIVE_DIR_DATED,
                                       **CONFIG0, verbose=True )
    
    dat['outputs'] =  outputs   
    logger.info(f'...finished in {(time()-t0)/60:.1f} min')
    fname = fname=f'{SUMMARY_OUTPUT_DIR_DATED}/{PREFIX}-{itime}.json'

    json.dump(dat, open(fname, 'w'), cls=NpEncoder)
    logger.info(f'Summary output written: {fname}')
    return dat
    

# loop it


In [50]:
result = run1()

Acquired settings from EPICS at: 2022-11-12T12:38:08-08:00
Acquired settings from EPICS at: 2022-11-12T12:38:08-08:00
Running evaluate_impact_with_distgen...
Running evaluate_impact_with_distgen...


/lscratch/jytang/tmp/ipykernel_812433/3072082356.py:17: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments.
  v = np.array(h5[k])


Setting distgen n_particle = 10000
Setting impact timeout = 10000
Setting impact header:Nx = 32
Setting impact header:Ny = 32
Setting impact header:Nz = 32
Setting impact numprocs = 64
Setting Npcol, Nprow = 8, 8
Enabling MPI
Setting impact command = ~/miniconda3/envs/lume-live-dev/bin/ImpactTexe
Setting impact command_mpi = ~/miniconda3/envs/lume-live-dev/bin/ImpactTexe-mpi
Setting impact mpi_run = salloc --partition milano --account ad:beamphysics -N 1 -n {nproc} mpirun -n {nproc} {command_mpi}
Setting impact stop = 14
Removed element: stop_1
Set stop to s = 14
Setting distgen t_dist:sigma_t:value = 6.794055201698514
Setting impact RFGUNB:rf_field_scale = 16486924.441374516
Setting impact RFGUNB:autophase_deg = 0.0
Setting impact SOL1B:solenoid_field_scale = 0.05449666295586827
Setting impact SOL2B:solenoid_field_scale = 0.028757730622052658
Setting distgen total_charge:value = 50.07263744
Setting impact BUN1B:rf_field_scale = 1696986.06875
Setting impact BUN1B:autophase_deg = -57.0


In [72]:
if __name__ == '__main__':
    while True:
        try:
            result = run1()
            sleep(10)
        except Exception as e:
            logger.info(e)
            if (e.__class__.__name__ == 'Exception'):
                logger.info('Stopping the Program')
                break
            else:
                logger.info('Something BAD happened. Sleeping for 10 s ...')      
                sleep(10)
            

Acquired settings from EPICS at: 2022-11-12T12:38:08-08:00
Acquired settings from EPICS at: 2022-11-12T12:38:08-08:00
Acquired settings from EPICS at: 2022-11-12T12:38:08-08:00
Running evaluate_impact_with_distgen...
Running evaluate_impact_with_distgen...
Running evaluate_impact_with_distgen...


/lscratch/jytang/tmp/ipykernel_1754105/3072082356.py:17: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments.
  v = np.array(h5[k])


Setting distgen n_particle = 10000
Setting impact timeout = 10000
Setting impact header:Nx = 32
Setting impact header:Ny = 32
Setting impact header:Nz = 32
Setting impact numprocs = 64
Setting Npcol, Nprow = 8, 8
Enabling MPI
Setting impact command = ~/miniconda3/envs/lume-live-dev/bin/ImpactTexe
Setting impact command_mpi = ~/miniconda3/envs/lume-live-dev/bin/ImpactTexe-mpi
Setting impact mpi_run = salloc --partition milano --account ad:beamphysics -N 1 -n {nproc} mpirun -n {nproc} {command_mpi}
Setting impact stop = 14
Removed element: stop_1
Set stop to s = 14
Setting distgen t_dist:sigma_t:value = 6.794055201698514
Setting impact RFGUNB:rf_field_scale = 16486924.441374516
Setting impact RFGUNB:autophase_deg = 0.0
Setting impact SOL1B:solenoid_field_scale = 0.05449666295586827
Setting impact SOL2B:solenoid_field_scale = 0.028757730622052658
Setting distgen total_charge:value = 50.07263744
Setting impact BUN1B:rf_field_scale = 1786301.125
Setting impact BUN1B:autophase_deg = -57.0
Se

KeyboardInterrupt: 